# Angular Momentum

### **Introduction**


### **What is it?**


### **Example**


# Simulation

## Setup

Installing packages (for Google Colab). If this notebook is opened in Google Colab then some packages must be installed to run the code!\
Then importing the scene and plot settings.

In [ ]:
#@title Run to install MuJoCo and `dm_control` for Google Colab

IS_COLAB = 'google.colab' in str(get_ipython())
if IS_COLAB:
    # download the repository
    !git clone https://github.com/commanderxa/alphalabs.git
    from alphalabs.mechanics.setup import install_packages_colab
    install_packages_colab()
    import alphalabs.mechanics.plot
    from alphalabs.mechanics.scene import Scene
else:
    import os, sys
    module_path = os.path.abspath(os.path.join(".."))
    if module_path not in sys.path:
        sys.path.append(module_path)
    # import the scene
    from scene import Scene
    import plot

## Import

Import all required packages to preform simulations. Packages include simulation engine, plotting libraries and other ones necessary for computations.

In [ ]:
%env MUJOCO_GL=egl

# simulation
from dm_control import mjcf

# for video recording
import mediapy

# computations
import numpy as np

# plot charts
import seaborn as sns
import matplotlib.pyplot as plt

## Initial Conditions

In this block constants are defined. They impact the environment, rendering and objects directly.

**Note**, don't set very high values for velocity as the simulation might crash. If you will experience such a situation, try reducing the velocity!

In [ ]:
# global
viscosity = 0.00002  # air resistance
density = 1.2  # air density

# collision constants
distance = 2 # distance [m]

mass = 1 # mass [kg]
v = 20  # speed [m/s]

# specifies whether to simulate the event with friction or without
is_ideal = True

# rendering
width = 1920
height = 1080
dpi = 600
duration = 30  # (seconds)
framerate = 60  # (Hz)

## Model

### Object

This class defines the object of our interest, a `box`. Here we write what is this object (box), what can it do (move, fall) and also add a camera that follows the object.

In [ ]:
class Box(object):

    def __init__(self, box_size: float, condim: int):
        self.model = mjcf.RootElement(model="box")
        self.worldbody = self.model.worldbody

        self.box_body = self.worldbody.add("body", name="box", pos=[0, 0, 0])
        self.box_geom = self.box_body.add(
            "geom",
            name="box",
            type="box",
            size=[box_size, box_size, box_size],
            rgba=[0.1, 0.3, 0.8, 1],
            mass=mass,
            condim=condim,
        )
        self.box_site = self.model.worldbody.add(
            "site",
            name="box_site",
            pos=[0, 0, 0],
            rgba=[0, 0, 0, 0],
            size=[0.001],
            type="sphere",
        )


class RotatingObject(object):
    """MuJoCo subsystem to simulate angular motion transfer via tendon."""

    def __init__(self):
        self.model = mjcf.RootElement(model="rotating_object")

        # Default sizes
        self.cylinder_radius = 0.25
        self.cylinder_height = 0.1
        self.box_size = 0.05
        self.tendon_length = 0.5

        self.worldbody = self.model.worldbody

        # Cylinder body (rotatable)
        self.cylinder_body = self.worldbody.add("body", name="cylinder", pos=[0, 0, 0])
        self.cylinder_geom = self.cylinder_body.add(
            "geom",
            name="base",
            type="cylinder",
            size=[self.cylinder_radius, self.cylinder_height],
            mass=mass,
            rgba=[0.6, 0.1, 0.1, 1],
            condim=1,
        )

        self.wrap_site = self.cylinder_body.add(
            "site",
            name="wrap_site",
            pos=[self.cylinder_radius, 0, 0],
            size=[0.001],
            rgba=[0, 0, 0, 0],
            type="sphere",
        )

        self.cylinder_joint = self.cylinder_body.add(
            "joint", name="cylinder_joint", type="hinge", axis=[0, 0, 1]
        )

        # Add actuator to rotate the cylinder
        self.actuator = self.model.actuator.add(
            "motor", name="motor", joint=self.cylinder_joint, gear="1"
        )

### World Model

Collecting everything into one general model.

In [ ]:
class Model(object):

    def __init__(self, condim: int) -> None:
        self.model = mjcf.RootElement(model="model")

        # set render info
        self.model.visual.__getattr__("global").offheight = height
        self.model.visual.__getattr__("global").offwidth = width

        # setting environment constants
        self.model.option.integrator = "RK4"
        self.model.option.flag.constraint = "enable"
        self.model.option.flag.contact = "enable"
        self.model.option.flag.gravity = "enable"
        self.model.option.flag.energy = "enable"

        if not is_ideal:
            self.model.option.viscosity = viscosity
            self.model.option.density = density
        else:
            self.model.option.viscosity = 0
            self.model.option.density = 0

        # create the environment (ground)
        self.arena = Scene(
            length=3 * distance,
            width=3 * distance,
            condim=condim,
            friction="0.001 0.005 0.0001",
        )
        self.arena_site = self.model.worldbody.add(
            "site", name="arena_site", pos=[0, 0, -0.1], rgba=[0, 0, 0, 0]
        )
        self.arena_site.attach(self.arena.model)

        # add object1
        self.obj = RotatingObject()
        obj_site = self.model.worldbody.add(
            "site",
            name="left_site",
            pos=[0, 0, 0],
            rgba=[0, 0, 0, 0],
        )
        obj_site.attach(self.obj.model)

        # Box body (will be pulled by tendon)
        self.box = Box(self.obj.box_size, condim=condim)
        self.attached_box = self.model.worldbody.attach(self.box.model)
        self.attached_box.add("joint", type="free")
        self.attached_box.pos = [distance, 0, 0.1]

        # Tendon path: from wrap site to box
        self.tendon = self.model.tendon.add(
            "spatial", name="tendon1", limited=True, range=f"0 {distance}", width="0.003"
        )
        self.tendon.add("site", site=self.obj.wrap_site)
        self.tendon.add("geom", geom="rotating_object/base")
        self.tendon.add("site", site=self.box.box_site)

        self.camera = self.model.worldbody.add(
            "camera",
            name="front",
            pos=[0, 0, distance * 3],
            euler=[0, 0, 0],
        )

## Simulation

Initializing the `physics` of the simulation.

In [ ]:
condim = 1 if is_ideal else 3
model = Model(condim=condim).model
physics = mjcf.Physics.from_mjcf_model(model)

First of all, the environment must be verified by rendering a picture.

In [ ]:
mediapy.show_image(physics.render(height, width, camera_id=0))

In [ ]:
mjcf.export_with_assets(model, "./", "angular.xml")

Next, it's time to make a simulation. This might take some time.

In [ ]:
physics.reset()

frames = []
timevals = []
velocity = []
position = []

physics.named.data.ctrl[0] = 1

while physics.time() < duration:
    # physics.named.data.qvel[0] = 10
    timevals.append(physics.time())
    velocity.append(physics.named.data.qvel["box/"][4:].copy())
    position.append(physics.named.data.geom_xpos["box/box"].copy())

    if len(frames) < physics.time() * framerate:
        frames.append(physics.render(height, width, camera_id=0))

    physics.step()

In [ ]:
mediapy.show_video(frames, fps=framerate)

Save the rendered video

In [ ]:
video_name = f"angular_rotation"
if is_ideal: video_name += "_ideal"
if not IS_COLAB: video_name = f"../../output/" + video_name
mediapy.write_video(video_name + ".mp4", images=frames, fps=framerate)

## Simulation Data Visualization

Convert data into numpy array to have more features

In [ ]:
timevals = np.array(timevals)
velocity = np.array(velocity)
position = np.array(position)

## Simulation Data Visualization

Collected velocities and position now can be plotted to investigate what happened to the object.

In [ ]:
cm = 1 / 2.54
figsize = (8.5 * cm, 3.75 * cm)
fig, ax = plt.subplots(ncols=2, figsize=figsize, dpi=dpi)
fig.subplots_adjust(wspace=1 * cm)

y_labels = ["Velocity, [m/s]", "Position, [m]"]
x_labels = ["Time, [s]", "Time, [s]"]
y_data = [[position[:, 0], position[:, 1]], [velocity[:, 0], velocity[:, 1]]]

for i in range(2):
    sns.lineplot(x=y_data[i][0], y=y_data[i][1], ax=ax[i], linewidth=1, color="blue")
    ax[i].set_ylabel(y_labels[i], fontsize=7, labelpad=2)
    ax[i].set_xlabel(x_labels[i], fontsize=7, labelpad=2)
    ax[i].tick_params(labelsize=6)
    ax[i].tick_params(which="both", direction="in", top=True, right=True, length=2)
    # ax[i].legend(fontsize=5, handlelength=1).get_frame().set_linewidth(0.5)

chart_name = f"angular_rotation"
if is_ideal: chart_name += "_ideal"
if not IS_COLAB: chart_name = f"../../output/" + chart_name
fig.savefig(chart_name + ".png", bbox_inches="tight")